# Linear Regression
Predicting Total Cost (USD)

## Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn import metrics
import warnings
warnings.filterwarnings('ignore')


## Load Cleaned Data

In [ ]:
df = pd.read_csv('../data/final_data.csv')
print('Shape:', df.shape)
display(df.head())


## Feature Engineering

In [ ]:
df['ship_date'] = pd.to_datetime(df['ship_date'])
df['delivery_date'] = pd.to_datetime(df['delivery_date'])
df['delivery_days'] = (df['delivery_date'] - df['ship_date']).dt.days
df['cost_per_kg'] = df['total_cost_usd'] / df['weight_kg']
df['volume_per_kg'] = df['volume_m3'] / df['weight_kg']
df['month_of_shipment'] = df['ship_date'].dt.month
df['year_of_shipment'] = df['ship_date'].dt.year


### Business Value of Features

| Feature | Description | Business Relevance |
|---------|-------------|-------------------|
| delivery_days | Time from shipment to delivery | Longer transit may increase handling costs |
| cost_per_kg | Cost per unit weight | Identifies high-value vs low-value shipments |
| volume_per_kg | Volumetric density | Bulky items consume more space, raising cost |
| month_of_shipment | Calendar month | Captures seasonal demand fluctuations |
| year_of_shipment | Calendar year | Captures annual inflation / rate trends |


## Prepare Features

In [ ]:
categorical_cols = df.select_dtypes(include='object').columns
for col in categorical_cols:
    df[col] = LabelEncoder().fit_transform(df[col].astype(str))

X = df.drop('total_cost_usd', axis=1).select_dtypes(include=[np.number])
y = df['total_cost_usd']
X = X.fillna(X.median())

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


## Train Linear Regression

In [ ]:
lr = LinearRegression()
lr.fit(X_train_scaled, y_train)
y_pred = lr.predict(X_test_scaled)

print('MAE:', metrics.mean_absolute_error(y_test, y_pred))
print('RMSE:', np.sqrt(metrics.mean_squared_error(y_test, y_pred)))
print('R2:', metrics.r2_score(y_test, y_pred))


## Visualize

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

ax1.scatter(y_test, y_pred, alpha=0.6)
ax1.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--')
ax1.set_xlabel('Actual')
ax1.set_ylabel('Predicted')
ax1.set_title('Actual vs Predicted')

residuals = y_test - y_pred
ax2.hist(residuals, bins=30, edgecolor='k', alpha=0.7)
ax2.set_xlabel('Residual')
ax2.set_ylabel('Frequency')
ax2.set_title('Residual Distribution')

plt.tight_layout()
plt.savefig('../model/lr_predictions.png', dpi=150, bbox_inches='tight')
plt.show()


## Coefficients

In [ ]:
coef_df = pd.DataFrame({'feature': X.columns, 'coefficient': lr.coef_})
coef_df['abs_coef'] = coef_df['coefficient'].abs()
top_features = coef_df.sort_values('abs_coef', ascending=False).head(10)
print('Top 10 features by coefficient magnitude:')
print(top_features[['feature', 'coefficient']].to_string(index=False))


## Save Model

In [ ]:
import joblib
joblib.dump(lr, '../model/linear_regression.pkl')
print('Model saved to ../model/linear_regression.pkl')


## Conclusion

The Linear Regression model was trained on engineered features including delivery_days, cost_per_kg, volume_per_kg, month_of_shipment, and year_of_shipment. Performance is summarized above in the MAE, RMSE, and R² metrics. The actual-vs-predicted plot and residual histogram provide visual diagnostics. Coefficients reveal which drivers most influence total cost, and the model artifact has been persisted for inference.
